In [1]:
import jax

jax.config.update("jax_cuda_visible_devices", "3")
# jax.config.update("jax_platform_name", "cpu")
# jax.disable_jit(disable=True)

jax.devices()

[CudaDevice(id=3)]

In [2]:
import jax

jax.print_environment_info()

jax:    0.6.2
jaxlib: 0.6.2
numpy:  2.3.0
python: 3.13.5 (main, Jun 12 2025, 12:40:22) [Clang 20.1.4 ]
device info: NVIDIA GeForce RTX 4090-1, 1 local devices"
process_count: 1
platform: uname_result(system='Linux', node='idsc-mifcom-1', release='6.8.0-59-generic', version='#61~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Apr 15 17:03:15 UTC 2', machine='x86_64')

$ nvidia-smi
Thu Jun 19 17:56:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.07             Driver Version: 570.133.07     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================

In [3]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from flax import nnx

# from hcnn.environment import Environment
from hcnn.optimization import OptimizationSolverShallow
from hcnn.policy.policy_mlp import PolicyMLP
from hcnn.projection import ProjectionNoSubstitutionNoTerminal, ProjectionPhi
from hcnn.system_constraints import SystemConstraintsBox
from hcnn.system_cost import SystemCostQuadratic
from hcnn.system_dynamics import SystemDynamicsLinear
from hcnn.training import DropoutMLP

In [4]:
state_matrix = jnp.array([[1.0, -1.0], [1.0, 0.0]])
input_matrix = jnp.array([[1.0], [0.0]])
sys_dynamics = SystemDynamicsLinear.factory(state_matrix, input_matrix)

In [5]:
sys_constraints = SystemConstraintsBox.factory(
    state_l=jnp.array([-2.0, -2.0]),
    state_u=jnp.array([2.0, 2.0]),
    action_l=jnp.array([-jnp.inf]),
    action_u=jnp.array([jnp.inf]),
)

In [6]:
state_cost = jnp.eye(2)
action_cost = jnp.eye(1)
sys_cost = SystemCostQuadratic.factory(state_cost, action_cost)

In [7]:
policy = PolicyMLP.factory(
    state_dim=sys_dynamics.state_dim,
    action_dim=sys_dynamics.action_dim,
    hidden_layer_sizes=[4] * 4,
    dropout_rate=0.15,
    rngs=nnx.Rngs(42),
)

In [8]:
problem = ProjectionNoSubstitutionNoTerminal.factory(
    policy=policy,
    sys_dynamics=sys_dynamics,
    sys_constraints=sys_constraints,
    horizon=20,
    relaxation_penalty=7.0,
)

phi = ProjectionPhi.factory(
    initial_state=jnp.array([1.0, 2.0]),
    policy=policy,
    jax_key=jax.random.key(0),
)

In [9]:
solver = OptimizationSolverShallow(max_iter=1_000, tol=0.1)

In [10]:
%time out = solver.solve(problem, phi).block_until_ready()

CPU times: user 2.31 s, sys: 221 ms, total: 2.53 s
Wall time: 2.2 s


In [11]:
%timeit solver.solve(problem, phi).block_until_ready()

92.8 ms ± 53.9 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [17]:
@jax.jit
def fun(initial_state):
    return solver.solve(problem, phi.replace(initial_state=initial_state))


jit_fn = jax.vmap(fun)

input = jax.random.normal(jax.random.key(4), (1024, 2))
jit_fn(input).block_until_ready()

%timeit jit_fn(input).block_until_ready()

175 ms ± 126 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


## ---  fix above first

In [ ]:
value_to_go = DropoutMLP(
    layer_sizes=[sys_dynamics.state_dim, 4, 1],
    dropout_rate=0.15,
    rngs=nnx.Rngs(10),
)

In [ ]:
relax_to_go = DropoutMLP(
    layer_sizes=[sys_dynamics.state_dim, 4, 1],
    dropout_rate=0.15,
    rngs=nnx.Rngs(10),
)

In [ ]:
environment = Environment(
    ## SYSTEM DEFINITION
    sys_constraints=sys_constraints,
    sys_cost=sys_cost,
    sys_dynamics=sys_dynamics,
    ## PROJECTION
    projection=projection,
    solver=solver,
)

In [ ]:
graphdef, state = environment.init(
    policy, value_to_go, relax_to_go, nnx.Rngs(42)
)

In [ ]:
for i in range(100):
    print(f"Iteration {i}")
    state = environment.train_step(graphdef, state)

In [ ]:
(
    environment,
    policy,
    policy_optimizer,
    value_to_go,
    value_to_go_optimizer,
    relax_to_go,
    relax_to_go_optimizer,
    metrics,
    rngs,
) = nnx.merge(graphdef, state)

In [ ]:
view_range = 6.0
samples_per_axis = 1_000
axis_samples = jnp.linspace(-view_range, view_range, samples_per_axis)
X, Y = jnp.meshgrid(axis_samples, axis_samples, indexing="xy")
states = jnp.stack((X.flatten(), Y.flatten()), axis=-1)

value_to_go.eval()
costs = nnx.vmap(lambda state: value_to_go(state, rngs=nnx.Rngs(0)))(
    states,
)
costs = costs.squeeze().reshape(samples_per_axis, samples_per_axis)

fig, ax = plt.subplots(figsize=(8, 6))
c = ax.contourf(
    states[:, 0].reshape(samples_per_axis, samples_per_axis),
    states[:, 1].reshape(samples_per_axis, samples_per_axis),
    costs,
    levels=20,
    cmap="viridis",
)
rect = plt.Rectangle(
    (-3, -3), 6, 6, linewidth=2, edgecolor="red", facecolor="none"
)
ax.add_patch(rect)
fig.colorbar(c, ax=ax, label="value-to-go")
ax.set_xlabel("State dimension 1")
ax.set_ylabel("State dimension 2")
ax.set_title("value-to-go over state")
plt.show()

In [ ]:
relax_to_go.eval()
costs = nnx.vmap(lambda state: relax_to_go(state, rngs=nnx.Rngs(0)))(
    states,
)
costs = costs.squeeze().reshape(samples_per_axis, samples_per_axis)

fig, ax = plt.subplots(figsize=(8, 6))
c = ax.contourf(
    states[:, 0].reshape(samples_per_axis, samples_per_axis),
    states[:, 1].reshape(samples_per_axis, samples_per_axis),
    costs,
    levels=20,
    cmap="viridis",
)
rect = plt.Rectangle(
    (-3, -3), 6, 6, linewidth=2, edgecolor="red", facecolor="none"
)
ax.add_patch(rect)
fig.colorbar(c, ax=ax, label="relax-to-go")
ax.set_xlabel("State dimension 1")
ax.set_ylabel("State dimension 2")
ax.set_title("relax-to-go over state")
plt.show()